In [ ]:
# @title
print("📦 Installing required packages...")
!pip install -q transformers datasets peft accelerate bitsandbytes trl sentencepiece protobuf
!pip install -q torch torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html


In [ ]:
# STEP 2: Import Libraries
# ============================================================================
print("\n📚 Importing libraries...")
import torch
import json
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import pandas as pd
from trl import SFTTrainer



📚 Importing libraries...


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [ ]:
# @title
print("\n🎨 Creating fashion recommendation dataset...")

# Sample training data - You should expand this with more examples
fashion_data = [
  {
    "instruction": "Recommend an outfit for a casual summer day",
    "input": "Person: 25 year old female, prefers comfortable clothing, loves pastel colors",
    "output": "I recommend a light pink linen shirt paired with white denim shorts and beige sandals. Add a straw hat and a small crossbody bag in cream color to complete the look."
  },
  {
    "instruction": "Suggest formal office wear",
    "input": "Person: 30 year old male, professional setting, prefers classic styles",
    "output": "A navy blue tailored suit with a crisp white dress shirt would be perfect. Pair it with a burgundy tie, black oxford shoes, and a matching leather belt."
  },
  {
    "instruction": "Create a date night outfit",
    "input": "Person: 28 year old female, romantic dinner, likes elegant styles",
    "output": "I suggest a black midi dress with a subtle floral pattern, paired with nude heels and minimal gold jewelry. Add a small clutch and a light cardigan for the evening."
  },
  {
    "instruction": "Recommend gym workout clothes",
    "input": "Person: 22 year old female, high-intensity training, prefers bold colors",
    "output": "Go for bright red high-waisted leggings with a black sports bra and a matching red cropped hoodie. Pair with white training sneakers and add a black gym bag."
  },
  {
    "instruction": "Suggest winter street style",
    "input": "Person: 26 year old male, urban lifestyle, loves streetwear",
    "output": "Layer a black hoodie under a gray puffer jacket, pair with dark blue jeans and white sneakers. Add a black beanie and a crossbody bag for the complete street look."
  },
  {
    "instruction": "Create a beach vacation outfit",
    "input": "Person: 24 year old female, tropical vacation, loves vibrant patterns",
    "output": "A colorful floral maxi dress in turquoise and coral tones, paired with tan leather sandals. Add oversized sunglasses, a woven beach bag, and a wide-brim straw hat."
  },
  {
    "instruction": "Recommend business casual attire",
    "input": "Person: 35 year old female, creative industry, prefers modern styles",
    "output": "Wear a camel-colored blazer over a white silk blouse with black tailored trousers. Pair with pointed-toe ankle boots and add a structured leather tote bag."
  },
  {
    "instruction": "Suggest weekend brunch outfit",
    "input": "Person: 27 year old male, relaxed setting, casual style",
    "output": "A light blue button-down shirt (sleeves rolled up) with khaki chinos and brown loafers. Add a brown leather belt and aviator sunglasses for a polished casual look."
  },
  {
    "instruction": "Create a concert outfit",
    "input": "Person: 23 year old female, rock concert, edgy style preferred",
    "output": "Black leather pants with a band graphic tee and a denim jacket. Pair with black combat boots, silver chain accessories, and a small black backpack."
  },
  {
    "instruction": "Recommend fall layering outfit",
    "input": "Person: 29 year old male, transitional weather, minimalist style",
    "output": "A charcoal gray crewneck sweater over a white t-shirt with dark wash jeans. Layer with a tan trench coat and finish with brown Chelsea boots."
  },
  {
    "instruction": "Suggest an outfit for a job interview",
    "input": "Person: 22 year old non-binary individual, seeking entry-level professional role, prefers smart and approachable styles",
    "output": "A tailored dark grey suit with a light blue button-up shirt or blouse. Pair with polished black dress shoes or loafers. Keep accessories minimal, perhaps a simple watch or small stud earrings."
  },
  {
    "instruction": "Create a cozy at-home lounge outfit",
    "input": "Person: 40 year old female, working from home, prioritizes comfort and warmth",
    "output": "Soft knit joggers in a neutral color like heather grey, paired with an oversized cream-colored cashmere sweater. Add fuzzy slippers and a pair of blue-light-blocking glasses."
  },
  {
    "instruction": "Recommend an outfit for a hiking trip",
    "input": "Person: 33 year old male, day hike in moderate weather, needs practical and durable gear",
    "output": "Moisture-wicking hiking pants, a breathable long-sleeved shirt (layer with a technical t-shirt), and a lightweight waterproof jacket. Finish with sturdy hiking boots, a backpack, and a baseball cap."
  },
  {
    "instruction": "Suggest a chic art gallery opening outfit",
    "input": "Person: 55 year old female, evening event, prefers sophisticated and unique pieces",
    "output": "A flowing dark emerald green jumpsuit with wide legs, paired with black block heels. Accessorize with a statement necklace or sculptural earrings and a sleek clutch."
  },
  {
    "instruction": "Create a festive holiday party look",
    "input": "Person: 29 year old female, office holiday party, likes subtle sparkle",
    "output": "A midi-length velvet dress in a rich jewel tone (like deep plum or sapphire), paired with metallic heels. Add delicate drop earrings and a small sequined clutch."
  },
  {
    "instruction": "Recommend an outfit for a music festival",
    "input": "Person: 20 year old male, outdoor festival, prefers bohemian and relaxed vibes",
    "output": "Distressed denim shorts, a graphic band tee, and an open flannel shirt. Pair with comfortable sneakers or canvas boots, a bucket hat, and a canvas backpack."
  },
  {
    "instruction": "Suggest a classic formal event ensemble",
    "input": "Person: 60 year old male, black-tie gala, appreciates timeless elegance",
    "output": "A classic black tuxedo with a white pleated dress shirt, a black bow tie, and black patent leather dress shoes. Finish with cufflinks and a pocket square."
  },
  {
    "instruction": "Create a travel day outfit",
    "input": "Person: 38 year old non-binary individual, long-haul flight, needs comfortable and presentable attire",
    "output": "Black comfortable travel trousers (like ponte knit), a soft long-sleeved t-shirt, and a stylish oversized cardigan or lightweight bomber jacket. Wear comfortable slip-on sneakers and carry a large tote bag."
  },
  {
    "instruction": "Recommend an outfit for a summer wedding guest",
    "input": "Person: 32 year old female, outdoor wedding, prefers light and airy styles",
    "output": "A flowy midi-dress in a pastel floral print or a solid soft hue like sky blue. Pair with block heels or dressy sandals, delicate jewelry, and a small clutch. Consider a wide-brimmed hat if the event is very sunny."
  },
  {
    "instruction": "Suggest a casual Friday office look",
    "input": "Person: 45 year old male, tech industry, values smart-casual and comfort",
    "output": "Dark wash slim-fit jeans, a well-fitting polo shirt or a fine-gauge knit sweater, and desert boots or stylish sneakers. Layer with a blazer if preferred for meetings."
  },
  {
    "instruction": "Create an outfit for a coffee shop study session",
    "input": "Person: 19 year old female, student, prefers comfortable and stylish everyday wear",
    "output": "High-waisted mom jeans, an oversized graphic sweatshirt, and white high-top sneakers. Add a cute beanie and a large canvas tote bag for books and laptop."
  },
  {
    "instruction": "Recommend an evening theatre outfit",
    "input": "Person: 50 year old female, classic play, appreciates sophisticated elegance",
    "output": "A tailored dark navy pantsuit with a silk camisole. Pair with pointed-toe heels and a delicate pearl necklace. A structured clutch would complete the look."
  },
  {
    "instruction": "Suggest outfit for a gardening session",
    "input": "Person: 65 year old male, backyard gardening, needs practical and durable clothes",
    "output": "Khaki work pants or sturdy jeans, a comfortable cotton t-shirt, and a lightweight flannel shirt as an outer layer. Wear work boots or sturdy garden clogs and a wide-brimmed sun hat."
  },
  {
    "instruction": "Create a vibrant summer party look",
    "input": "Person: 28 year old non-binary individual, outdoor summer party, loves bold colors and playful styles",
    "output": "A colorful Hawaiian-print button-up shirt (worn open over a plain tee or tied at the waist), high-waisted linen shorts, and bright canvas sneakers. Accessorize with beaded necklaces and quirky sunglasses."
  },
  {
    "instruction": "Recommend maternity wear for a baby shower",
    "input": "Person: 30 year old pregnant female, celebratory event, prefers elegant and comfortable maternity styles",
    "output": "A flowy midi-length maternity dress in a soft floral print or solid pastel color. Pair with low block heels or elegant flats, delicate jewelry, and a small shoulder bag."
  },
  {
    "instruction": "Suggest an outfit for a chilly spring morning run",
    "input": "Person: 35 year old male, morning jog, needs athletic and warm layers",
    "output": "Moisture-wicking running tights, a long-sleeved technical running top, and a lightweight windbreaker jacket. Add a running beanie or headband and gloves if it's very cold. Finish with running shoes."
  },
  {
    "instruction": "Create a sophisticated business dinner outfit",
    "input": "Person: 42 year old female, executive dinner, seeks polished and authoritative style",
    "output": "A well-fitted black sheath dress with a structured blazer in a complementary color (e.g., charcoal grey or deep plum). Pair with elegant pumps and a sophisticated watch. A subtle pendant necklace would be suitable."
  },
  {
    "instruction": "Recommend an outfit for a casual barbecue",
    "input": "Person: 27 year old male, backyard gathering, prefers relaxed and easy-going attire",
    "output": "Denim shorts, a graphic tee or a simple polo shirt, and comfortable canvas sneakers. A baseball cap could be added for sun protection."
  },
  {
    "instruction": "Suggest an outfit for a winter wedding guest",
    "input": "Person: 30 year old female, formal indoor wedding, desires warm and elegant attire",
    "output": "A long-sleeved velvet midi dress in a deep jewel tone, paired with closed-toe heels or elegant ankle boots. Add a faux fur stole or a tailored wool coat for warmth. Statement earrings would be lovely."
  },
  {
    "instruction": "Create an outfit for a university lecture",
    "input": "Person: 20 year old male, student, likes practical and cool casual wear",
    "output": "Dark wash jeans, a graphic t-shirt, and an open plaid flannel shirt. Pair with classic sneakers and a sturdy backpack. A simple beanie can add a touch of style."
  },
  {
    "instruction": "Recommend an outfit for a high-tea event",
    "input": "Person: 60 year old female, elegant afternoon tea, prefers classic and refined styles",
    "output": "A sophisticated A-line dress in a delicate floral or pastel print, or a tailored skirt suit. Pair with low heels or elegant flats, a pearl necklace, and perhaps a fascinator or elegant hat. A small clutch is ideal."
  },
  {
    "instruction": "Suggest an outfit for grocery shopping",
    "input": "Person: 35 year old non-binary individual, quick errands, prioritizes comfort and ease",
    "output": "Comfortable leggings or track pants, an oversized hoodie or sweatshirt, and slip-on sneakers. A large reusable shopping bag and sunglasses are practical additions."
  },
  {
    "instruction": "Create an outfit for a photography shoot (casual outdoor)",
    "input": "Person: 24 year old female, natural look, likes soft and earthy tones",
    "output": "A flowy white or cream-colored linen dress, paired with simple flat sandals. Keep makeup minimal and hair natural. A delicate necklace or small hoop earrings would be suitable."
  },
  {
    "instruction": "Recommend an outfit for a corporate presentation",
    "input": "Person: 48 year old male, senior executive, needs authoritative and polished attire",
    "output": "A charcoal grey or navy pinstripe suit, a crisp white dress shirt, and a power tie (e.g., solid red or deep blue). Pair with highly polished black leather oxford shoes and a luxury watch."
  },
  {
    "instruction": "Suggest an outfit for a fun day at an amusement park",
    "input": "Person: 21 year old female, active and playful, needs comfortable and practical clothing",
    "output": "Comfortable denim shorts or jeans, a fun graphic t-shirt, and supportive sneakers. A small crossbody bag for essentials, sunglasses, and a hat for sun protection are key."
  },
  {
    "instruction": "Create an outfit for a networking event",
    "input": "Person: 30 year old male, professional but social, wants to be approachable yet sharp",
    "output": "Smart dark wash jeans or chinos, a well-fitted sport coat in navy or grey, and a button-down shirt (patterned or solid). Pair with loafers or dressy sneakers. A nice watch can complete the look."
  },
  {
    "instruction": "Recommend an outfit for a relaxing spa day",
    "input": "Person: 55 year old female, seeking ultimate comfort, wants easy-to-wear items",
    "output": "Soft cotton loungewear set (top and pants) in a calming neutral color. Pair with comfortable slip-on sandals or soft slippers. Bring a lightweight robe for extra comfort."
  },
  {
    "instruction": "Suggest an outfit for a cultural festival",
    "input": "Person: 26 year old non-binary individual, outdoor event, loves unique and expressive styles",
    "output": "Patterned wide-leg trousers or a flowing maxi skirt, paired with a fitted tank top or bandeau. Layer with a kimono or embroidered jacket. Add comfortable sandals or espadrilles, statement jewelry, and a decorative headscarf."
  },
  {
    "instruction": "Create a professional headshot outfit",
    "input": "Person: 38 year old female, needs a confident and approachable image",
    "output": "A well-fitting blazer (solid color like navy, black, or grey) over a simple blouse or shell top in a flattering color. Keep accessories minimal and elegant, such as stud earrings or a delicate necklace."
  },
  {
    "instruction": "Recommend an outfit for a camping trip",
    "input": "Person: 40 year old male, multi-day camping, needs durable and layered clothing",
    "output": "Convertible hiking pants, a long-sleeved base layer, a fleece jacket, and a waterproof shell jacket. Wear sturdy hiking boots and wool socks. A beanie for colder nights and a baseball cap for sun are essential."
  },
  {
    "instruction": "Suggest an outfit for a casual dinner with friends",
    "input": "Person: 23 year old female, relaxed atmosphere, prefers chic and comfortable",
    "output": "Dark wash skinny jeans, a silky camisole, and an open knit cardigan or a faux leather jacket. Pair with ankle boots or stylish flats. A small crossbody bag completes the look."
  },
  {
    "instruction": "Create an outfit for a museum visit",
    "input": "Person: 50 year old female, wants to be comfortable yet stylish for walking",
    "output": "Tailored wide-leg trousers in a neutral color, a fitted long-sleeved top, and a lightweight structured blazer. Pair with comfortable yet elegant loafers or low-heeled ankle boots. A large tote bag for essentials."
  },
  {
    "instruction": "Recommend an outfit for a sporting event (stadium)",
    "input": "Person: 30 year old male, cheering for a team, needs comfortable and spirited attire",
    "output": "Your favorite team jersey or t-shirt, comfortable jeans, and sneakers. A baseball cap in team colors and a light jacket (if cool) are practical. A fanny pack or small backpack for essentials."
  },
  {
    "instruction": "Suggest an outfit for a fancy cocktail party",
    "input": "Person: 29 year old female, evening event, desires glamorous and modern",
    "output": "A sophisticated cocktail dress in a luxurious fabric like satin or silk, with unique detailing. Pair with strappy heels and a statement clutch. Bold earrings or a delicate necklace would be perfect."
  },
  {
    "instruction": "Create an outfit for a day trip to the city",
    "input": "Person: 25 year old non-binary individual, sightseeing and exploring, needs practical yet fashionable attire",
    "output": "Comfortable but stylish sneakers, dark wash jeans or tailored chinos, a striped t-shirt, and a stylish trench coat or denim jacket. A spacious backpack or crossbody bag for essentials and a scarf for versatility."
  },
  {
    "instruction": "Recommend an outfit for an afternoon picnic",
    "input": "Person: 22 year old female, casual outdoor gathering, likes cute and comfortable",
    "output": "A gingham or floral print sundress with comfortable flat sandals or white sneakers. A denim jacket can be added for warmth. A woven basket bag and a wide-brimmed hat complete the charming look."
  },
  {
    "instruction": "Suggest an outfit for a business casual lunch meeting",
    "input": "Person: 40 year old male, client meeting, needs polished yet relaxed",
    "output": "Chinos in navy or stone color, a crisp button-down shirt (untucked or semi-tucked), and a V-neck sweater or a lightweight sport coat. Pair with loafers or clean leather sneakers."
  },
  {
    "instruction": "Create an outfit for a home renovation project",
    "input": "Person: 50 year old male, DIY work, needs durable and disposable clothing",
    "output": "Old jeans or work trousers, an old t-shirt, and a sturdy long-sleeved work shirt or flannel. Wear work boots and thick socks. A baseball cap to keep hair out of the way and safety glasses are crucial."
  },
  {
    "instruction": "Recommend an outfit for a book club gathering",
    "input": "Person: 35 year old female, relaxed social event, prefers cozy and smart casual",
    "output": "Dark wash jeans or corduroy pants, a comfortable knit sweater or a long-sleeved top, and ankle boots or stylish flats. A delicate necklace and glasses could add to the academic-chic vibe."
  },
  {
    "instruction": "Suggest an outfit for a rainy day",
    "input": "Person: 28 year old female, needs to stay dry and stylish, urban environment",
    "output": "Waterproof trench coat, skinny jeans or dark trousers, and waterproof ankle boots or stylish rain boots. An umbrella and a waterproof handbag are essential accessories."
  },
  {
    "instruction": "Create an outfit for a volunteer event (outdoors)",
    "input": "Person: 60 year old female, community service, needs practical and comfortable",
    "output": "Sturdy canvas trousers or dark jeans, a comfortable long-sleeved t-shirt, and a light fleece or utility vest. Wear comfortable walking shoes or boots. A sun hat and work gloves are recommended."
  },
  {
    "instruction": "Recommend an outfit for a formal awards ceremony",
    "input": "Person: 45 year old non-binary individual, black-tie optional, wants to make a statement",
    "output": "A well-tailored velvet suit in a deep color like forest green or burgundy, paired with a silk camisole or a crisp white dress shirt (no tie). Polished dress shoes or elegant loafers. A statement brooch or pocket square can elevate the look."
  },
  {
    "instruction": "Suggest an outfit for a casual movie night at home",
    "input": "Person: 20 year old male, relaxing with friends, prioritizes ultimate comfort",
    "output": "Soft sweatpants or basketball shorts, an oversized hoodie or a comfortable t-shirt. Add fluffy socks or house slippers. Keep it simple and cozy."
  },
  {
    "instruction": "Create an outfit for a market visit",
    "input": "Person: 30 year old female, farmers market, prefers bohemian and relaxed",
    "output": "A flowy maxi skirt with a simple white tank top, layered with a denim vest or light kimono. Comfortable flat sandals or espadrilles. A large tote bag for purchases and a straw hat for sun protection."
  },
  {
    "instruction": "Recommend an outfit for a professional photo shoot (corporate)",
    "input": "Person: 35 year old male, needs a smart and trustworthy image",
    "output": "A well-fitted navy or grey suit, a crisp light blue or white dress shirt, and a classic silk tie. Polished leather dress shoes and a good quality watch are key. Ensure the suit is tailored."
  },
  {
    "instruction": "Suggest an outfit for a leisurely bike ride",
    "input": "Person: 27 year old female, casual ride, needs comfortable and movement-friendly attire",
    "output": "Athletic leggings or comfortable shorts, a breathable t-shirt or tank top, and a lightweight hoodie or jacket. Wear sneakers and a bike helmet. A small backpack or fanny pack for essentials."
  },
  {
    "instruction": "Create an outfit for a family gathering (holiday)",
    "input": "Person: 50 year old female, festive but comfortable, wants to look put-together",
    "output": "A festive knit sweater or a blouse in a holiday color, paired with comfortable dark trousers or a midi skirt. Low block heels or elegant flats. Add some festive jewelry like a subtle brooch or earrings."
  },
  {
    "instruction": "Recommend an outfit for a first date (casual)",
    "input": "Person: 28 year old male, coffee date, wants to appear approachable and stylish",
    "output": "Dark wash slim-fit jeans, a well-fitting crewneck t-shirt (plain or subtle graphic), and a smart bomber jacket or an open chambray shirt. Clean white sneakers or stylish chukka boots complete the look."
  },
  {
    "instruction": "Suggest an outfit for a bachelorette party (night out)",
    "input": "Person: 25 year old female, lively celebration, desires fun and glamorous",
    "output": "A sparkly mini dress or a stylish jumpsuit with cut-outs, paired with strappy heels. Add bold jewelry like chandelier earrings and a small statement clutch. Consider a fun hair accessory."
  },
  {
    "instruction": "Create an outfit for a home office setup",
    "input": "Person: 32 year old non-binary individual, needs comfortable yet presentable for video calls",
    "output": "Comfortable, tailored joggers or dark wash jeans paired with a collared shirt or a stylish knit top. A simple blazer or cardigan can be added for video calls. Wear comfortable slippers or clean indoor shoes."
  },
  {
    "instruction": "Recommend an outfit for a charity run/walk event",
    "input": "Person: 40 year old female, participates for fun, needs comfortable athletic wear",
    "output": "Athletic leggings or shorts, a moisture-wicking t-shirt (often provided by the event), and comfortable running shoes. A baseball cap and sunglasses for sun protection. A fanny pack for essentials."
  },
  {
    "instruction": "Suggest an outfit for a school play/recital",
    "input": "Person: 38 year old male, parent attending, wants to look neat and supportive",
    "output": "Dark wash jeans or chinos, a neat polo shirt or a casual button-down shirt. Layer with a V-neck sweater or a casual blazer. Comfortable loafers or clean sneakers."
  },
  {
    "instruction": "Create an outfit for a glamorous gala",
    "input": "Person: 60 year old female, black-tie event, prefers timeless and luxurious",
    "output": "A floor-length evening gown in a rich fabric like silk or velvet, in a classic color (black, navy, or deep red). Pair with elegant heels, a sophisticated clutch, and fine jewelry (diamonds or pearls)."
  },
  {
    "instruction": "Recommend an outfit for a productive day at the library",
    "input": "Person: 20 year old male, long study session, prioritizes comfort and focus",
    "output": "Comfortable jeans or cargo pants, a plain t-shirt, and an oversized hoodie or a soft flannel shirt. Wear sneakers or comfortable slip-ons. A backpack for books and laptop."
  },
  {
    "instruction": "Suggest an outfit for a casual evening out (drinks)",
    "input": "Person: 26 year old female, catching up with friends, wants chic and relaxed",
    "output": "High-waisted dark wash jeans, a stylish top (e.g., silk cami, off-the-shoulder blouse), and a faux leather jacket. Pair with ankle boots or fashionable sneakers. A crossbody bag."
  },
  {
    "instruction": "Create an outfit for a photography studio session (glamour)",
    "input": "Person: 30 year old female, wants a dramatic and elegant look",
    "output": "A floor-length gown with a high slit or interesting back detail, in a rich color. Pair with high heels and statement jewelry. Focus on dramatic makeup and a polished hairstyle."
  },
  {
    "instruction": "Recommend an outfit for a pet show/competition",
    "input": "Person: 40 year old female, needs comfortable and practical for moving around with animals",
    "output": "Comfortable jeans or sturdy chinos, a durable polo shirt or a plain t-shirt, and a lightweight utility vest with pockets. Wear comfortable walking shoes or sneakers. A baseball cap and a small bag for pet treats."
  },
  {
    "instruction": "Suggest an outfit for a rainy autumn day",
    "input": "Person: 35 year old male, needs warmth and water protection, urban commuting",
    "output": "Waterproof trench coat or a stylish parka, dark wash jeans or chinos, and waterproof leather boots. A warm knit sweater or fleece under the coat. An umbrella and a durable messenger bag."
  },
  {
    "instruction": "Create an outfit for a corporate retreat (team building)",
    "input": "Person: 45 year old non-binary individual, needs comfortable yet professional attire for activities",
    "output": "Smart dark wash jeans or tailored casual trousers, a professional polo shirt or a button-down shirt, and a lightweight blazer or smart cardigan. Comfortable yet presentable sneakers or loafers."
  },
  {
    "instruction": "Recommend an outfit for a trip to the zoo",
    "input": "Person: 28 year old female, walking a lot, wants comfort and fun",
    "output": "Comfortable shorts or capris, a fun graphic t-shirt or a bright tank top, and supportive sneakers. A small backpack, sunglasses, and a sun hat are essential for a day of walking outdoors."
  },
  {
    "instruction": "Suggest an outfit for a fancy dinner at a restaurant",
    "input": "Person: 32 year old male, celebrating an anniversary, wants elegant and impressive",
    "output": "A tailored dark grey or navy suit (or a sport coat with dress trousers), a crisp white dress shirt, and a stylish tie (patterned or solid silk). Polished leather dress shoes and a good watch."
  },
  {
    "instruction": "Create an outfit for a children's birthday party (active)",
    "input": "Person: 30 year old female, needs to move and play, wants casual and durable",
    "output": "Comfortable jeans or leggings, a simple t-shirt or a casual blouse, and sneakers. A denim jacket can be added if it's cool. Keep accessories minimal for practicality."
  },
  {
    "instruction": "Recommend an outfit for a winter holiday market",
    "input": "Person: 25 year old female, outdoor event, needs warmth and festive style",
    "output": "Warm leggings or thermal tights under jeans, a cozy chunky knit sweater, and a stylish wool coat or a puffer jacket. Add a festive scarf, a knit beanie, warm gloves, and waterproof ankle boots."
  },
  {
    "instruction": "Suggest an outfit for a live podcast recording",
    "input": "Person: 35 year old male, wants to look smart but approachable for an audience",
    "output": "Dark chinos or smart jeans, a well-fitting button-down shirt (perhaps a subtle pattern), and a lightweight blazer or a stylish cardigan. Polished loafers or neat casual shoes."
  },
  {
    "instruction": "Create an outfit for a cooking class",
    "input": "Person: 40 year old female, needs practical and comfortable attire, won't get too messy",
    "output": "Comfortable jeans or dark trousers, a simple t-shirt or a fitted long-sleeved top (avoiding loose sleeves). Wear comfortable flats or sneakers. An apron will likely be provided, but a hair tie is useful."
  },
  {
    "instruction": "Recommend an outfit for a theme park (Halloween event)",
    "input": "Person: 20 year old female, wants a fun costume, but also comfortable for walking",
    "output": "A fun, simple costume (e.g., witch hat with black leggings and a themed top, or animal ears with appropriate colored clothes) paired with comfortable sneakers. A small backpack and a light jacket for the evening."
  },
  {
    "instruction": "Suggest an outfit for a grand opening ceremony",
    "input": "Person: 50 year old male, needs to look authoritative and celebratory",
    "output": "A well-tailored dark suit (e.g., navy or charcoal), a crisp white dress shirt, and a sophisticated tie (perhaps a power color or a subtle pattern). Polished dress shoes and a pocket square for a touch of flair."
  },
  {
    "instruction": "Create an outfit for a relaxing boat ride/cruise (daytime)",
    "input": "Person: 65 year old female, casual sightseeing, wants comfort and sun protection",
    "output": "White linen trousers or capris, a nautical-striped t-shirt or a light tunic, and comfortable flat sandals or deck shoes. A wide-brimmed sun hat, sunglasses, and a light cardigan for sea breezes."
  },
  {
    "instruction": "Recommend an outfit for a DIY workshop",
    "input": "Person: 30 year old non-binary individual, learning new skills, needs practical and sturdy clothing",
    "output": "Durable jeans or cargo pants, a comfortable cotton t-shirt, and a long-sleeved work shirt or flannel. Wear sturdy closed-toe shoes or work boots. Consider wearing an old apron or bringing one."
  },
  {
    "instruction": "Suggest an outfit for a job fair",
    "input": "Person: 22 year old female, looking for internships, wants to make a good first impression",
    "output": "Tailored trousers or a knee-length pencil skirt, a conservative blouse (button-up or shell top), and a well-fitted blazer. Low heels or polished flats. A neat portfolio or tote bag for resumes."
  },
  {
    "instruction": "Create an outfit for a casual housewarming party",
    "input": "Person: 35 year old male, social gathering, wants to be relaxed but put-together",
    "output": "Chinos or dark jeans, a smart polo shirt or a casual button-down, and clean sneakers or loafers. A lightweight sweater or a casual blazer can be added for a slightly dressier touch."
  },
  {
    "instruction": "Recommend an outfit for a yoga or Pilates class",
    "input": "Person: 40 year old female, needs flexible and comfortable activewear",
    "output": "Form-fitting yoga leggings, a breathable tank top or sports bra, and a light cover-up or hoodie for before/after class. Barefoot or non-slip grip socks."
  },
  {
    "instruction": "Suggest an outfit for a winter sporting event (ice hockey)",
    "input": "Person: 28 year old male, spectator, needs to stay warm in a cold arena",
    "output": "Warm layered clothing: thermal base layer, a thick long-sleeved shirt, a warm hoodie or sweater, and a heavy winter coat. Insulated pants or thick jeans. Warm hat, gloves, and winter boots."
  },
  {
    "instruction": "Create an outfit for a book signing event",
    "input": "Person: 50 year old female, meeting an author, prefers intellectual and chic",
    "output": "A stylish midi skirt (pleated or A-line) with a fitted knit top or a silk blouse. Pair with elegant ankle boots or sophisticated flats. A tailored blazer can add polish. Carry a chic handbag."
  },
  {
    "instruction": "Recommend an outfit for a local parade",
    "input": "Person: 20 year old male, outdoor spectator, wants comfortable and casual",
    "output": "Comfortable jeans or shorts, a graphic t-shirt, and a hoodie or light jacket depending on weather. Sneakers, a baseball cap, and sunglasses are practical additions. A small backpack for water and snacks."
  },
  {
    "instruction": "Suggest an outfit for a weekend getaway (city break)",
    "input": "Person: 30 year old female, exploring and dining, needs versatile and stylish",
    "output": "Dark wash skinny jeans, a versatile long-sleeved top or sweater, a stylish trench coat or wool jacket. Comfortable yet chic ankle boots or fashionable sneakers. A crossbody bag and a large scarf."
  },
  {
    "instruction": "Create an outfit for a volunteer shift (indoor)",
    "input": "Person: 65 year old female, needs comfortable and practical for light work",
    "output": "Comfortable dark trousers or jeans, a simple long-sleeved t-shirt or a soft sweater. Comfortable walking shoes or flats. Keep jewelry minimal and hair tied back if necessary."
  },
  {
    "instruction": "Recommend an outfit for a spring garden party",
    "input": "Person: 35 year old non-binary individual, elegant outdoor event, prefers sophisticated florals",
    "output": "A floral print midi dress or a stylish jumpsuit in a light fabric. Pair with elegant espadrille wedges or block heels. A light cardigan or shawl. Accessorize with a wide-brimmed hat or a delicate fascinator."
  },
  {
    "instruction": "Suggest an outfit for a professional conference",
    "input": "Person: 48 year old male, presenting, needs to look polished and confident",
    "output": "A well-tailored dark suit (navy, charcoal, or black), a crisp white or light blue dress shirt, and a tie. Polished leather dress shoes. Ensure the suit is wrinkle-free. A good quality leather briefcase."
  },
  {
    "instruction": "Recommend an outfit for a casual summer day",
    "input": "Person: 25 year old female, prefers comfortable clothing, loves pastel colors",
    "output": "I recommend a light pink linen shirt paired with white denim shorts and beige sandals. Add a straw hat and a small crossbody bag in cream color to complete the look."
  },
  {
    "instruction": "Suggest formal office wear",
    "input": "Person: 30 year old male, professional setting, prefers classic styles",
    "output": "A navy blue tailored suit with a crisp white dress shirt would be perfect. Pair it with a burgundy tie, black oxford shoes, and a matching leather belt."
  },
  {
    "instruction": "Create a date night outfit",
    "input": "Person: 28 year old female, romantic dinner, likes elegant styles",
    "output": "I suggest a black midi dress with a subtle floral pattern, paired with nude heels and minimal gold jewelry. Add a small clutch and a light cardigan for the evening."
  },
  {
    "instruction": "Recommend gym workout clothes",
    "input": "Person: 22 year old female, high-intensity training, prefers bold colors",
    "output": "Go for bright red high-waisted leggings with a black sports bra and a matching red cropped hoodie. Pair with white training sneakers and add a black gym bag."
  },
  {
    "instruction": "Suggest winter street style",
    "input": "Person: 26 year old male, urban lifestyle, loves streetwear",
    "output": "Layer a black hoodie under a gray puffer jacket, pair with dark blue jeans and white sneakers. Add a black beanie and a crossbody bag for the complete street look."
  },
  {
    "instruction": "Create a beach vacation outfit",
    "input": "Person: 24 year old female, tropical vacation, loves vibrant patterns",
    "output": "A colorful floral maxi dress in turquoise and coral tones, paired with tan leather sandals. Add oversized sunglasses, a woven beach bag, and a wide-brim straw hat."
  },
  {
    "instruction": "Recommend business casual attire",
    "input": "Person: 35 year old female, creative industry, prefers modern styles",
    "output": "Wear a camel-colored blazer over a white silk blouse with black tailored trousers. Pair with pointed-toe ankle boots and add a structured leather tote bag."
  },
  {
    "instruction": "Suggest weekend brunch outfit",
    "input": "Person: 27 year old male, relaxed setting, casual style",
    "output": "A light blue button-down shirt (sleeves rolled up) with khaki chinos and brown loafers. Add a brown leather belt and aviator sunglasses for a polished casual look."
  },
  {
    "instruction": "Create a concert outfit",
    "input": "Person: 23 year old female, rock concert, edgy style preferred",
    "output": "Black leather pants with a band graphic tee and a denim jacket. Pair with black combat boots, silver chain accessories, and a small black backpack."
  },
  {
    "instruction": "Recommend fall layering outfit",
    "input": "Person: 29 year old male, transitional weather, minimalist style",
    "output": "A charcoal gray crewneck sweater over a white t-shirt with dark wash jeans. Layer with a tan trench coat and finish with brown Chelsea boots."
  },
  {
    "instruction": "Suggest an outfit for a job interview",
    "input": "Person: 22 year old non-binary individual, seeking entry-level professional role, prefers smart and approachable styles",
    "output": "A tailored dark grey suit with a light blue button-up shirt or blouse. Pair with polished black dress shoes or loafers. Keep accessories minimal, perhaps a simple watch or small stud earrings."
  },
  {
    "instruction": "Create a cozy at-home lounge outfit",
    "input": "Person: 40 year old female, working from home, prioritizes comfort and warmth",
    "output": "Soft knit joggers in a neutral color like heather grey, paired with an oversized cream-colored cashmere sweater. Add fuzzy slippers and a pair of blue-light-blocking glasses."
  },
  {
    "instruction": "Recommend an outfit for a hiking trip",
    "input": "Person: 33 year old male, day hike in moderate weather, needs practical and durable gear",
    "output": "Moisture-wicking hiking pants, a breathable long-sleeved shirt (layer with a technical t-shirt), and a lightweight waterproof jacket. Finish with sturdy hiking boots, a backpack, and a baseball cap."
  },
  {
    "instruction": "Suggest a chic art gallery opening outfit",
    "input": "Person: 55 year old female, evening event, prefers sophisticated and unique pieces",
    "output": "A flowing dark emerald green jumpsuit with wide legs, paired with black block heels. Accessorize with a statement necklace or sculptural earrings and a sleek clutch."
  },
  {
    "instruction": "Create a festive holiday party look",
    "input": "Person: 29 year old female, office holiday party, likes subtle sparkle",
    "output": "A midi-length velvet dress in a rich jewel tone (like deep plum or sapphire), paired with metallic heels. Add delicate drop earrings and a small sequined clutch."
  },
  {
    "instruction": "Recommend an outfit for a music festival",
    "input": "Person: 20 year old male, outdoor festival, prefers bohemian and relaxed vibes",
    "output": "Distressed denim shorts, a graphic band tee, and an open flannel shirt. Pair with comfortable sneakers or canvas boots, a bucket hat, and a canvas backpack."
  },
  {
    "instruction": "Suggest a classic formal event ensemble",
    "input": "Person: 60 year old male, black-tie gala, appreciates timeless elegance",
    "output": "A classic black tuxedo with a white pleated dress shirt, a black bow tie, and black patent leather dress shoes. Finish with cufflinks and a pocket square."
  },
  {
    "instruction": "Create a travel day outfit",
    "input": "Person: 38 year old non-binary individual, long-haul flight, needs comfortable and presentable attire",
    "output": "Black comfortable travel trousers (like ponte knit), a soft long-sleeved t-shirt, and a stylish oversized cardigan or lightweight bomber jacket. Wear comfortable slip-on sneakers and carry a large tote bag."
  },
  {
    "instruction": "Recommend an outfit for a summer wedding guest",
    "input": "Person: 32 year old female, outdoor wedding, prefers light and airy styles",
    "output": "A flowy midi-dress in a pastel floral print or a solid soft hue like sky blue. Pair with block heels or dressy sandals, delicate jewelry, and a small clutch. Consider a wide-brimmed hat if the event is very sunny."
  },
  {
    "instruction": "Suggest a casual Friday office look",
    "input": "Person: 45 year old male, tech industry, values smart-casual and comfort",
    "output": "Dark wash slim-fit jeans, a well-fitting polo shirt or a fine-gauge knit sweater, and desert boots or stylish sneakers. Layer with a blazer if preferred for meetings."
  },
  {
    "instruction": "Create an outfit for a coffee shop study session",
    "input": "Person: 19 year old female, student, prefers comfortable and stylish everyday wear",
    "output": "High-waisted mom jeans, an oversized graphic sweatshirt, and white high-top sneakers. Add a cute beanie and a large canvas tote bag for books and laptop."
  },
  {
    "instruction": "Recommend an evening theatre outfit",
    "input": "Person: 50 year old female, classic play, appreciates sophisticated elegance",
    "output": "A tailored dark navy pantsuit with a silk camisole. Pair with pointed-toe heels and a delicate pearl necklace. A structured clutch would complete the look."
  },
  {
    "instruction": "Suggest outfit for a gardening session",
    "input": "Person: 65 year old male, backyard gardening, needs practical and durable clothes",
    "output": "Khaki work pants or sturdy jeans, a comfortable cotton t-shirt, and a lightweight flannel shirt as an outer layer. Wear work boots or sturdy garden clogs and a wide-brimmed sun hat."
  },
  {
    "instruction": "Create a vibrant summer party look",
    "input": "Person: 28 year old non-binary individual, outdoor summer party, loves bold colors and playful styles",
    "output": "A colorful Hawaiian-print button-up shirt (worn open over a plain tee or tied at the waist), high-waisted linen shorts, and bright canvas sneakers. Accessorize with beaded necklaces and quirky sunglasses."
  },
  {
    "instruction": "Recommend maternity wear for a baby shower",
    "input": "Person: 30 year old pregnant female, celebratory event, prefers elegant and comfortable maternity styles",
    "output": "A flowy midi-length maternity dress in a soft floral print or solid pastel color. Pair with low block heels or elegant flats, delicate jewelry, and a small shoulder bag."
  },
  {
    "instruction": "Suggest an outfit for a chilly spring morning run",
    "input": "Person: 35 year old male, morning jog, needs athletic and warm layers",
    "output": "Moisture-wicking running tights, a long-sleeved technical running top, and a lightweight windbreaker jacket. Add a running beanie or headband and gloves if it's very cold. Finish with running shoes."
  },
  {
    "instruction": "Create a sophisticated business dinner outfit",
    "input": "Person: 42 year old female, executive dinner, seeks polished and authoritative style",
    "output": "A well-fitted black sheath dress with a structured blazer in a complementary color (e.g., charcoal grey or deep plum). Pair with elegant pumps and a sophisticated watch. A subtle pendant necklace would be suitable."
  },
  {
    "instruction": "Recommend an outfit for a casual barbecue",
    "input": "Person: 27 year old male, backyard gathering, prefers relaxed and easy-going attire",
    "output": "Denim shorts, a graphic tee or a simple polo shirt, and comfortable canvas sneakers. A baseball cap could be added for sun protection."
  },
  {
    "instruction": "Suggest an outfit for a winter wedding guest",
    "input": "Person: 30 year old female, formal indoor wedding, desires warm and elegant attire",
    "output": "A long-sleeved velvet midi dress in a deep jewel tone, paired with closed-toe heels or elegant ankle boots. Add a faux fur stole or a tailored wool coat for warmth. Statement earrings would be lovely."
  },
  {
    "instruction": "Create an outfit for a university lecture",
    "input": "Person: 20 year old male, student, likes practical and cool casual wear",
    "output": "Dark wash jeans, a graphic t-shirt, and an open plaid flannel shirt. Pair with classic sneakers and a sturdy backpack. A simple beanie can add a touch of style."
  },
  {
    "instruction": "Recommend an outfit for a high-tea event",
    "input": "Person: 60 year old female, elegant afternoon tea, prefers classic and refined styles",
    "output": "A sophisticated A-line dress in a delicate floral or pastel print, or a tailored skirt suit. Pair with low heels or elegant flats, a pearl necklace, and perhaps a fascinator or elegant hat. A small clutch is ideal."
  },
  {
    "instruction": "Suggest an outfit for grocery shopping",
    "input": "Person: 35 year old non-binary individual, quick errands, prioritizes comfort and ease",
    "output": "Comfortable leggings or track pants, an oversized hoodie or sweatshirt, and slip-on sneakers. A large reusable shopping bag and sunglasses are practical additions."
  },
  {
    "instruction": "Create an outfit for a photography shoot (casual outdoor)",
    "input": "Person: 24 year old female, natural look, likes soft and earthy tones",
    "output": "A flowy white or cream-colored linen dress, paired with simple flat sandals. Keep makeup minimal and hair natural. A delicate necklace or small hoop earrings would be suitable."
  },
  {
    "instruction": "Recommend an outfit for a corporate presentation",
    "input": "Person: 48 year old male, senior executive, needs authoritative and polished attire",
    "output": "A charcoal grey or navy pinstripe suit, a crisp white dress shirt, and a power tie (e.g., solid red or deep blue). Pair with highly polished black leather oxford shoes and a luxury watch."
  },
  {
    "instruction": "Suggest an outfit for a fun day at an amusement park",
    "input": "Person: 21 year old female, active and playful, needs comfortable and practical clothing",
    "output": "Comfortable denim shorts or jeans, a fun graphic t-shirt, and supportive sneakers. A small crossbody bag for essentials, sunglasses, and a hat for sun protection are key."
  },
  {
    "instruction": "Create an outfit for a networking event",
    "input": "Person: 30 year old male, professional but social, wants to be approachable yet sharp",
    "output": "Smart dark wash jeans or chinos, a well-fitted sport coat in navy or grey, and a button-down shirt (patterned or solid). Pair with loafers or dressy sneakers. A nice watch can complete the look."
  },
  {
    "instruction": "Recommend an outfit for a relaxing spa day",
    "input": "Person: 55 year old female, seeking ultimate comfort, wants easy-to-wear items",
    "output": "Soft cotton loungewear set (top and pants) in a calming neutral color. Pair with comfortable slip-on sandals or soft slippers. Bring a lightweight robe for extra comfort."
  },
  {
    "instruction": "Suggest an outfit for a cultural festival",
    "input": "Person: 26 year old non-binary individual, outdoor event, loves unique and expressive styles",
    "output": "Patterned wide-leg trousers or a flowing maxi skirt, paired with a fitted tank top or bandeau. Layer with a kimono or embroidered jacket. Add comfortable sandals or espadrilles, statement jewelry, and a decorative headscarf."
  },
  {
    "instruction": "Create a professional headshot outfit",
    "input": "Person: 38 year old female, needs a confident and approachable image",
    "output": "A well-fitting blazer (solid color like navy, black, or grey) over a simple blouse or shell top in a flattering color. Keep accessories minimal and elegant, such as stud earrings or a delicate necklace."
  },
  {
    "instruction": "Recommend an outfit for a camping trip",
    "input": "Person: 40 year old male, multi-day camping, needs durable and layered clothing",
    "output": "Convertible hiking pants, a long-sleeved base layer, a fleece jacket, and a waterproof shell jacket. Wear sturdy hiking boots and wool socks. A beanie for colder nights and a baseball cap for sun are essential."
  },
  {
    "instruction": "Suggest an outfit for a casual dinner with friends",
    "input": "Person: 23 year old female, relaxed atmosphere, prefers chic and comfortable",
    "output": "Dark wash skinny jeans, a silky camisole, and an open knit cardigan or a faux leather jacket. Pair with ankle boots or stylish flats. A small crossbody bag completes the look."
  },
  {
    "instruction": "Create an outfit for a museum visit",
    "input": "Person: 50 year old female, wants to be comfortable yet stylish for walking",
    "output": "Tailored wide-leg trousers in a neutral color, a fitted long-sleeved top, and a lightweight structured blazer. Pair with comfortable yet elegant loafers or low-heeled ankle boots. A large tote bag for essentials."
  },
  {
    "instruction": "Recommend an outfit for a sporting event (stadium)",
    "input": "Person: 30 year old male, cheering for a team, needs comfortable and spirited attire",
    "output": "Your favorite team jersey or t-shirt, comfortable jeans, and sneakers. A baseball cap in team colors and a light jacket (if cool) are practical. A fanny pack or small backpack for essentials."
  },
  {
    "instruction": "Suggest an outfit for a fancy cocktail party",
    "input": "Person: 29 year old female, evening event, desires glamorous and modern",
    "output": "A sophisticated cocktail dress in a luxurious fabric like satin or silk, with unique detailing. Pair with strappy heels and a statement clutch. Bold earrings or a delicate necklace would be perfect."
  },
  {
    "instruction": "Create an outfit for a day trip to the city",
    "input": "Person: 25 year old non-binary individual, sightseeing and exploring, needs practical yet fashionable attire",
    "output": "Comfortable but stylish sneakers, dark wash jeans or tailored chinos, a striped t-shirt, and a stylish trench coat or denim jacket. A spacious backpack or crossbody bag for essentials and a scarf for versatility."
  },
  {
    "instruction": "Recommend an outfit for an afternoon picnic",
    "input": "Person: 22 year old female, casual outdoor gathering, likes cute and comfortable",
    "output": "A gingham or floral print sundress with comfortable flat sandals or white sneakers. A denim jacket can be added for warmth. A woven basket bag and a wide-brimmed hat complete the charming look."
  },
  {
    "instruction": "Suggest an outfit for a business casual lunch meeting",
    "input": "Person: 40 year old male, client meeting, needs polished yet relaxed",
    "output": "Chinos in navy or stone color, a crisp button-down shirt (untucked or semi-tucked), and a V-neck sweater or a lightweight sport coat. Pair with loafers or clean leather sneakers."
  },
  {
    "instruction": "Create an outfit for a home renovation project",
    "input": "Person: 50 year old male, DIY work, needs durable and disposable clothing",
    "output": "Old jeans or work trousers, an old t-shirt, and a sturdy long-sleeved work shirt or flannel. Wear work boots and thick socks. A baseball cap to keep hair out of the way and safety glasses are crucial."
  },
  {
    "instruction": "Recommend an outfit for a book club gathering",
    "input": "Person: 35 year old female, relaxed social event, prefers cozy and smart casual",
    "output": "Dark wash jeans or corduroy pants, a comfortable knit sweater or a long-sleeved top, and ankle boots or stylish flats. A delicate necklace and glasses could add to the academic-chic vibe."
  },
  {
    "instruction": "Suggest an outfit for a rainy day",
    "input": "Person: 28 year old female, needs to stay dry and stylish, urban environment",
    "output": "Waterproof trench coat, skinny jeans or dark trousers, and waterproof ankle boots or stylish rain boots. An umbrella and a waterproof handbag are essential accessories."
  },
  {
    "instruction": "Create an outfit for a volunteer event (outdoors)",
    "input": "Person: 60 year old female, community service, needs practical and comfortable",
    "output": "Sturdy canvas trousers or dark jeans, a comfortable long-sleeved t-shirt, and a light fleece or utility vest. Wear comfortable walking shoes or boots. A sun hat and work gloves are recommended."
  },
  {
    "instruction": "Recommend an outfit for a formal awards ceremony",
    "input": "Person: 45 year old non-binary individual, black-tie optional, wants to make a statement",
    "output": "A well-tailored velvet suit in a deep color like forest green or burgundy, paired with a silk camisole or a crisp white dress shirt (no tie). Polished dress shoes or elegant loafers. A statement brooch or pocket square can elevate the look."
  },
  {
    "instruction": "Suggest an outfit for a casual movie night at home",
    "input": "Person: 20 year old male, relaxing with friends, prioritizes ultimate comfort",
    "output": "Soft sweatpants or basketball shorts, an oversized hoodie or a comfortable t-shirt. Add fluffy socks or house slippers. Keep it simple and cozy."
  },
  {
    "instruction": "Create an outfit for a market visit",
    "input": "Person: 30 year old female, farmers market, prefers bohemian and relaxed",
    "output": "A flowy maxi skirt with a simple white tank top, layered with a denim vest or light kimono. Comfortable flat sandals or espadrilles. A large tote bag for purchases and a straw hat for sun protection."
  },
  {
    "instruction": "Recommend an outfit for a professional photo shoot (corporate)",
    "input": "Person: 35 year old male, needs a smart and trustworthy image",
    "output": "A well-fitted navy or grey suit, a crisp light blue or white dress shirt, and a classic silk tie. Polished leather dress shoes and a good quality watch are key. Ensure the suit is tailored."
  },
  {
    "instruction": "Suggest an outfit for a leisurely bike ride",
    "input": "Person: 27 year old female, casual ride, needs comfortable and movement-friendly attire",
    "output": "Athletic leggings or comfortable shorts, a breathable t-shirt or tank top, and a lightweight hoodie or jacket. Wear sneakers and a bike helmet. A small backpack or fanny pack for essentials."
  },
  {
    "instruction": "Create an outfit for a family gathering (holiday)",
    "input": "Person: 50 year old female, festive but comfortable, wants to look put-together",
    "output": "A festive knit sweater or a blouse in a holiday color, paired with comfortable dark trousers or a midi skirt. Low block heels or elegant flats. Add some festive jewelry like a subtle brooch or earrings."
  },
  {
    "instruction": "Recommend an outfit for a first date (casual)",
    "input": "Person: 28 year old male, coffee date, wants to appear approachable and stylish",
    "output": "Dark wash slim-fit jeans, a well-fitting crewneck t-shirt (plain or subtle graphic), and a smart bomber jacket or an open chambray shirt. Clean white sneakers or stylish chukka boots complete the look."
  },
  {
    "instruction": "Suggest an outfit for a bachelorette party (night out)",
    "input": "Person: 25 year old female, lively celebration, desires fun and glamorous",
    "output": "A sparkly mini dress or a stylish jumpsuit with cut-outs, paired with strappy heels. Add bold jewelry like chandelier earrings and a small statement clutch. Consider a fun hair accessory."
  },
  {
    "instruction": "Create an outfit for a home office setup",
    "input": "Person: 32 year old non-binary individual, needs comfortable yet presentable for video calls",
    "output": "Comfortable, tailored joggers or dark wash jeans paired with a collared shirt or a stylish knit top. A simple blazer or cardigan can be added for video calls. Wear comfortable slippers or clean indoor shoes."
  },
  {
    "instruction": "Recommend an outfit for a charity run/walk event",
    "input": "Person: 40 year old female, participates for fun, needs comfortable athletic wear",
    "output": "Athletic leggings or shorts, a moisture-wicking t-shirt (often provided by the event), and comfortable running shoes. A baseball cap and sunglasses for sun protection. A fanny pack for essentials."
  },
  {
    "instruction": "Suggest an outfit for a school play/recital",
    "input": "Person: 38 year old male, parent attending, wants to look neat and supportive",
    "output": "Dark wash jeans or chinos, a neat polo shirt or a casual button-down shirt. Layer with a V-neck sweater or a casual blazer. Comfortable loafers or clean sneakers."
  },
  {
    "instruction": "Create an outfit for a glamorous gala",
    "input": "Person: 60 year old female, black-tie event, prefers timeless and luxurious",
    "output": "A floor-length evening gown in a rich fabric like silk or velvet, in a classic color (black, navy, or deep red). Pair with elegant heels, a sophisticated clutch, and fine jewelry (diamonds or pearls)."
  },
  {
    "instruction": "Recommend an outfit for a productive day at the library",
    "input": "Person: 20 year old male, long study session, prioritizes comfort and focus",
    "output": "Comfortable jeans or cargo pants, a plain t-shirt, and an oversized hoodie or a soft flannel shirt. Wear sneakers or comfortable slip-ons. A backpack for books and laptop."
  },
  {
    "instruction": "Suggest an outfit for a casual evening out (drinks)",
    "input": "Person: 26 year old female, catching up with friends, wants chic and relaxed",
    "output": "High-waisted dark wash jeans, a stylish top (e.g., silk cami, off-the-shoulder blouse), and a faux leather jacket. Pair with ankle boots or fashionable sneakers. A crossbody bag."
  },
  {
    "instruction": "Create an outfit for a photography studio session (glamour)",
    "input": "Person: 30 year old female, wants a dramatic and elegant look",
    "output": "A floor-length gown with a high slit or interesting back detail, in a rich color. Pair with high heels and statement jewelry. Focus on dramatic makeup and a polished hairstyle."
  },
  {
    "instruction": "Recommend an outfit for a pet show/competition",
    "input": "Person: 40 year old female, needs comfortable and practical for moving around with animals",
    "output": "Comfortable jeans or sturdy chinos, a durable polo shirt or a plain t-shirt, and a lightweight utility vest with pockets. Wear comfortable walking shoes or sneakers. A baseball cap and a small bag for pet treats."
  },
  {
    "instruction": "Suggest an outfit for a rainy autumn day",
    "input": "Person: 35 year old male, needs warmth and water protection, urban commuting",
    "output": "Waterproof trench coat or a stylish parka, dark wash jeans or chinos, and waterproof leather boots. A warm knit sweater or fleece under the coat. An umbrella and a durable messenger bag."
  },
  {
    "instruction": "Create an outfit for a corporate retreat (team building)",
    "input": "Person: 45 year old non-binary individual, needs comfortable yet professional attire for activities",
    "output": "Smart dark wash jeans or tailored casual trousers, a professional polo shirt or a button-down shirt, and a lightweight blazer or smart cardigan. Comfortable yet presentable sneakers or loafers."
  },
  {
    "instruction": "Recommend an outfit for a trip to the zoo",
    "input": "Person: 28 year old female, walking a lot, wants comfort and fun",
    "output": "Comfortable shorts or capris, a fun graphic t-shirt or a bright tank top, and supportive sneakers. A small backpack, sunglasses, and a sun hat are essential for a day of walking outdoors."
  },
  {
    "instruction": "Suggest an outfit for a fancy dinner at a restaurant",
    "input": "Person: 32 year old male, celebrating an anniversary, wants elegant and impressive",
    "output": "A tailored dark grey or navy suit (or a sport coat with dress trousers), a crisp white dress shirt, and a stylish tie (patterned or solid silk). Polished leather dress shoes and a good watch."
  },
  {
    "instruction": "Create an outfit for a children's birthday party (active)",
    "input": "Person: 30 year old female, needs to move and play, wants casual and durable",
    "output": "Comfortable jeans or leggings, a simple t-shirt or a casual blouse, and sneakers. A denim jacket can be added if it's cool. Keep accessories minimal for practicality."
  },
  {
    "instruction": "Recommend an outfit for a winter holiday market",
    "input": "Person: 25 year old female, outdoor event, needs warmth and festive style",
    "output": "Warm leggings or thermal tights under jeans, a cozy chunky knit sweater, and a stylish wool coat or a puffer jacket. Add a festive scarf, a knit beanie, warm gloves, and waterproof ankle boots."
  },
  {
    "instruction": "Suggest an outfit for a live podcast recording",
    "input": "Person: 35 year old male, wants to look smart but approachable for an audience",
    "output": "Dark chinos or smart jeans, a well-fitting button-down shirt (perhaps a subtle pattern), and a lightweight blazer or a stylish cardigan. Polished loafers or neat casual shoes."
  },
  {
    "instruction": "Create an outfit for a cooking class",
    "input": "Person: 40 year old female, needs practical and comfortable attire, won't get too messy",
    "output": "Comfortable jeans or dark trousers, a simple t-shirt or a fitted long-sleeved top (avoiding loose sleeves). Wear comfortable flats or sneakers. An apron will likely be provided, but a hair tie is useful."
  },
  {
    "instruction": "Recommend an outfit for a theme park (Halloween event)",
    "input": "Person: 20 year old female, wants a fun costume, but also comfortable for walking",
    "output": "A fun, simple costume (e.g., witch hat with black leggings and a themed top, or animal ears with appropriate colored clothes) paired with comfortable sneakers. A small backpack and a light jacket for the evening."
  },
  {
    "instruction": "Suggest an outfit for a grand opening ceremony",
    "input": "Person: 50 year old male, needs to look authoritative and celebratory",
    "output": "A well-tailored dark suit (e.g., navy or charcoal), a crisp white dress shirt, and a sophisticated tie (perhaps a power color or a subtle pattern). Polished dress shoes and a pocket square for a touch of flair."
  },
  {
    "instruction": "Create an outfit for a relaxing boat ride/cruise (daytime)",
    "input": "Person: 65 year old female, casual sightseeing, wants comfort and sun protection",
    "output": "White linen trousers or capris, a nautical-striped t-shirt or a light tunic, and comfortable flat sandals or deck shoes. A wide-brimmed sun hat, sunglasses, and a light cardigan for sea breezes."
  },
  {
    "instruction": "Recommend an outfit for a DIY workshop",
    "input": "Person: 30 year old non-binary individual, learning new skills, needs practical and sturdy clothing",
    "output": "Durable jeans or cargo pants, a comfortable cotton t-shirt, and a long-sleeved work shirt or flannel. Wear sturdy closed-toe shoes or work boots. Consider wearing an old apron or bringing one."
  },
  {
    "instruction": "Suggest an outfit for a job fair",
    "input": "Person: 22 year old female, looking for internships, wants to make a good first impression",
    "output": "Tailored trousers or a knee-length pencil skirt, a conservative blouse (button-up or shell top), and a well-fitted blazer. Low heels or polished flats. A neat portfolio or tote bag for resumes."
  },
  {
    "instruction": "Create an outfit for a casual housewarming party",
    "input": "Person: 35 year old male, social gathering, wants to be relaxed but put-together",
    "output": "Chinos or dark jeans, a smart polo shirt or a casual button-down, and clean sneakers or loafers. A lightweight sweater or a casual blazer can be added for a slightly dressier touch."
  },
  {
    "instruction": "Recommend an outfit for a yoga or Pilates class",
    "input": "Person: 40 year old female, needs flexible and comfortable activewear",
    "output": "Form-fitting yoga leggings, a breathable tank top or sports bra, and a light cover-up or hoodie for before/after class. Barefoot or non-slip grip socks."
  },
  {
    "instruction": "Suggest an outfit for a winter sporting event (ice hockey)",
    "input": "Person: 28 year old male, spectator, needs to stay warm in a cold arena",
    "output": "Warm layered clothing: thermal base layer, a thick long-sleeved shirt, a warm hoodie or sweater, and a heavy winter coat. Insulated pants or thick jeans. Warm hat, gloves, and winter boots."
  },
  {
    "instruction": "Create an outfit for a book signing event",
    "input": "Person: 50 year old female, meeting an author, prefers intellectual and chic",
    "output": "A stylish midi skirt (pleated or A-line) with a fitted knit top or a silk blouse. Pair with elegant ankle boots or sophisticated flats. A tailored blazer can add polish. Carry a chic handbag."
  },
  {
    "instruction": "Recommend an outfit for a local parade",
    "input": "Person: 20 year old male, outdoor spectator, wants comfortable and casual",
    "output": "Comfortable jeans or shorts, a graphic t-shirt, and a hoodie or light jacket depending on weather. Sneakers, a baseball cap, and sunglasses are practical additions. A small backpack for water and snacks."
  },
  {
    "instruction": "Suggest an outfit for a weekend getaway (city break)",
    "input": "Person: 30 year old female, exploring and dining, needs versatile and stylish",
    "output": "Dark wash skinny jeans, a versatile long-sleeved top or sweater, a stylish trench coat or wool jacket. Comfortable yet chic ankle boots or fashionable sneakers. A crossbody bag and a large scarf."
  },
  {
    "instruction": "Create an outfit for a volunteer shift (indoor)",
    "input": "Person: 65 year old female, needs comfortable and practical for light work",
    "output": "Comfortable dark trousers or jeans, a simple long-sleeved t-shirt or a soft sweater. Comfortable walking shoes or flats. Keep jewelry minimal and hair tied back if necessary."
  },
  {
    "instruction": "Recommend an outfit for a spring garden party",
    "input": "Person: 35 year old non-binary individual, elegant outdoor event, prefers sophisticated florals",
    "output": "A floral print midi dress or a stylish jumpsuit in a light fabric. Pair with elegant espadrille wedges or block heels. A light cardigan or shawl. Accessorize with a wide-brimmed hat or a delicate fascinator."
  },
  {
    "instruction": "Suggest an outfit for a professional conference",
    "input": "Person: 48 year old male, presenting, needs to look polished and confident",
    "output": "A well-tailored dark suit (navy, charcoal, or black), a crisp white or light blue dress shirt, and a tie. Polished leather dress shoes. Ensure the suit is wrinkle-free. A good quality leather briefcase."
  },
  {
    "instruction": "Recommend an outfit for a chilly autumn day",
    "input": "Person: 28 year old female, prefers cozy and stylish, enjoys earthy tones",
    "output": "High-waisted dark wash jeans, a cream cable-knit sweater, and brown ankle boots. Layer with a camel-colored trench coat and add a plaid scarf."
  },
  {
    "instruction": "Suggest outfit for a warm spring afternoon",
    "input": "Person: 35 year old male, casual outdoor event, likes relaxed comfort",
    "output": "Light wash jeans, a short-sleeved linen shirt in a pastel color, and white canvas sneakers. Add a pair of aviator sunglasses for a cool touch."
  },
  {
    "instruction": "Create an outfit for a rainy winter commute",
    "input": "Person: 40 year old non-binary individual, urban professional, needs functional and sleek",
    "output": "Waterproof trench coat, black tailored trousers, a merino wool sweater, and waterproof Chelsea boots. Carry a durable umbrella and a leather laptop bag."
  },
  {
    "instruction": "Recommend activewear for a hot summer run",
    "input": "Person: 22 year old female, high-intensity training, needs breathable fabrics",
    "output": "Moisture-wicking cycling shorts, a lightweight mesh tank top, and supportive running shoes. A high-impact sports bra and a visor for sun protection are essential."
  },
  {
    "instruction": "Suggest a look for a snowy holiday gathering",
    "input": "Person: 50 year old male, family event, prefers smart casual and warm",
    "output": "Dark corduroy pants, a festive Fair Isle sweater, and warm leather chukka boots. Layer with a thick wool overcoat and a cashmere scarf. Add leather gloves."
  },
  {
    "instruction": "Create an outfit for a breezy beach evening",
    "input": "Person: 24 year old female, romantic walk, likes flowy and comfortable",
    "output": "A maxi sundress in a light fabric, paired with flat sandals. Add a light denim jacket or a linen kimono for warmth. A small clutch and simple jewelry."
  },
  {
    "instruction": "Recommend attire for a humid summer office day",
    "input": "Person: 30 year old female, corporate environment, needs breathable professional wear",
    "output": "A lightweight sleeveless sheath dress in a breathable fabric like cotton or linen blend. Pair with closed-toe pumps and minimal jewelry. A light blazer can be added for meetings."
  },
  {
    "instruction": "Suggest an outfit for a cold weather outdoor concert",
    "input": "Person: 26 year old male, rock music fan, wants edgy and warm",
    "output": "Thermal base layers under dark jeans, a band t-shirt, and an oversized flannel. Layer with a heavy-duty parka or a thick leather jacket. Add a beanie, gloves, and sturdy combat boots."
  },
  {
    "instruction": "Create an ensemble for a mild autumn wedding",
    "input": "Person: 32 year old non-binary individual, guest, seeks elegant and unique",
    "output": "A rich emerald green velvet midi dress or a tailored jumpsuit, paired with elegant block heels. A delicate wrap or pashmina for the evening. Minimal gold jewelry."
  },
  {
    "instruction": "Recommend an outfit for a very hot outdoor festival",
    "input": "Person: 20 year old female, music lover, needs practical and cool",
    "output": "Denim cut-off shorts, a cropped tank top or a bralette, and comfortable flat sandals or sneakers. A wide-brimmed straw hat, sunglasses, and a small fanny pack for essentials. Hydration pack is key!"
  },
  {
    "instruction": "Suggest a look for a damp spring forest walk",
    "input": "Person: 45 year old male, nature enthusiast, needs practical and waterproof",
    "output": "Water-resistant hiking pants, a breathable long-sleeved top, and a lightweight waterproof jacket. Wear waterproof hiking boots and a baseball cap. A small backpack for water and snacks."
  },
  {
    "instruction": "Create an outfit for a freezing winter night out",
    "input": "Person: 29 year old female, fancy dinner, wants glamorous and warm",
    "output": "A long-sleeved velvet jumpsuit or a luxurious midi dress in a dark color. Layer with a faux fur coat or a thick wool wrap. Pair with elegant ankle boots and warm tights. Add statement earrings."
  },
  {
    "instruction": "Recommend attire for a sunny, breezy golf game",
    "input": "Person: 60 year old male, avid golfer, prefers classic and comfortable",
    "output": "Golf polo shirt, tailored golf trousers or shorts, and comfortable golf shoes. A light windbreaker jacket, a golf visor or cap, and sunglasses are essential. Golf glove."
  },
  {
    "instruction": "Suggest a look for a misty autumn morning market",
    "input": "Person: 30 year old female, weekend errands, likes rustic chic",
    "output": "Comfortable dark jeans, a chunky knit sweater, and waterproof ankle boots. Layer with a waxed cotton jacket. A large tote bag for purchases and a warm beanie."
  },
  {
    "instruction": "Create an outfit for a sweltering summer city break",
    "input": "Person: 25 year old male, sightseeing, needs light and breathable",
    "output": "Linen shorts, a short-sleeved button-up shirt in a light fabric (e.g., seersucker), and comfortable leather sandals or espadrilles. Sunglasses and a fedora hat for sun protection."
  },
  {
    "instruction": "Recommend attire for a mild winter evening bonfire",
    "input": "Person: 20 year old non-binary individual, casual outdoor gathering, wants relaxed and warm",
    "output": "Thick corduroy pants, an oversized flannel shirt, and a hooded fleece jacket. Wear sturdy high-top sneakers or boots. A knit beanie and fingerless gloves."
  },
  {
    "instruction": "Suggest a look for a spring shower at a kids' park",
    "input": "Person: 35 year old female, parent, needs practical and protective",
    "output": "Waterproof rain jacket, comfortable dark wash jeans or leggings, and waterproof sneakers or wellington boots. A baseball cap or hooded jacket to keep hair dry. Minimal accessories for ease of movement."
  },
  {
    "instruction": "Create an outfit for a hot and dry desert hike",
    "input": "Person: 40 year old male, adventurous, needs protective and cooling",
    "output": "Lightweight, long-sleeved UPF-rated hiking shirt, convertible hiking pants, and sturdy hiking boots. A wide-brimmed hat, sunglasses, and a hydration pack are crucial for sun and heat protection."
  },
  {
    "instruction": "Recommend attire for a breezy summer picnic",
    "input": "Person: 28 year old female, casual outdoor meal, likes feminine and comfortable",
    "output": "A flowy midi skirt in a floral print, a simple white camisole, and flat sandals. Add a light denim jacket or a linen wrap for when the breeze picks up. A straw hat and a basket bag."
  },
  {
    "instruction": "Suggest a look for a freezing cold indoor event",
    "input": "Person: 55 year old female, ice show spectator, needs warmth and comfort",
    "output": "Warm tailored trousers, a cashmere turtleneck sweater, and elegant ankle boots with thick socks. Layer with a stylish wool coat and add a warm scarf and gloves."
  },
  {
    "instruction": "Create an outfit for a sunny autumn vineyard tour",
    "input": "Person: 40 year old non-binary individual, leisurely outing, wants chic and comfortable",
    "output": "Tailored dark wash jeans, a soft knit sweater in a rich autumnal color, and comfortable leather ankle boots. Layer with a stylish blazer or a long cardigan. A scarf and sunglasses."
  },
  {
    "instruction": "Recommend activewear for a cold winter hike",
    "input": "Person: 33 year old male, strenuous activity, needs warm layers",
    "output": "Thermal base layers (top and bottom), insulated hiking pants, a fleece mid-layer, and a waterproof/windproof outer shell jacket. Winter hiking boots, wool socks, a beanie, and insulated gloves."
  },
  {
    "instruction": "Suggest a look for a humid summer evening wedding",
    "input": "Person: 30 year old female, guest, desires elegant and lightweight",
    "output": "A flowing empire-waist maxi dress in a breathable silk or chiffon, in a bright solid color or tropical print. Pair with elegant flat sandals or low wedges. Delicate jewelry and an updo hairstyle."
  },
  {
    "instruction": "Create an outfit for a blustery spring coastal walk",
    "input": "Person: 25 year old male, outdoor adventure, needs wind protection",
    "output": "Durable jeans or cargo pants, a long-sleeved t-shirt, and a windproof softshell jacket. Wear sturdy walking shoes or sneakers. A knit beanie and a scarf for extra warmth against the wind."
  },
  {
    "instruction": "Recommend attire for a rainy autumn wedding",
    "input": "Person: 38 year old non-binary individual, indoor event, wants formal and waterproof",
    "output": "A tailored dark suit in a wool blend, a crisp dress shirt, and a festive tie or patterned scarf. Pair with polished waterproof dress shoes. A classic trench coat for arrival and departure."
  },
  {
    "instruction": "Suggest a look for a scorching summer beach day",
    "input": "Person: 22 year old female, relaxing, needs minimal and cool",
    "output": "A stylish one-piece swimsuit or bikini, paired with a sheer cover-up or high-waisted linen shorts. Wide-brimmed straw hat, oversized sunglasses, and comfortable flip-flops. Beach bag and sunscreen are vital."
  },
  {
    "instruction": "Create an outfit for a mild winter outdoor market",
    "input": "Person: 50 year old female, browsing crafts, wants comfortable and chic",
    "output": "Dark wash straight-leg jeans, a fine-gauge knit turtleneck, and a stylish wool coat. Pair with comfortable ankle boots. Add a cashmere scarf and elegant leather gloves."
  },
  {
    "instruction": "Recommend attire for a sunny spring city exploration",
    "input": "Person: 27 year old male, sightseeing, needs stylish and walkable",
    "output": "Light wash slim-fit jeans, a classic striped t-shirt, and a light denim jacket or a bomber jacket. Clean white sneakers or comfortable loafers. Sunglasses and a stylish backpack for essentials."
  },
  {
    "instruction": "Suggest a look for a cold, indoor sports game",
    "input": "Person: 30 year old female, basketball game, wants spirited and warm",
    "output": "Team jersey layered over a long-sleeved top, comfortable dark jeans, and warm sneakers. A thick team scarf, a knit beanie, and a cozy puffer vest or jacket to stay warm in the arena."
  },
  {
    "instruction": "Create an outfit for a very humid tropical vacation dinner",
    "input": "Person: 35 year old male, upscale resort, needs elegant and breathable",
    "output": "Linen blend trousers in a light color, a short-sleeved camp collar shirt in a tropical print or solid bright color. Leather sandals or loafers. A lightweight watch."
  },
  {
    "instruction": "Recommend attire for a blustery autumn photoshoot",
    "input": "Person: 24 year old female, outdoor session, likes ethereal and layered",
    "output": "A flowing maxi dress in an autumnal color (e.g., burnt orange, deep green), layered with a chunky knit cardigan. Add a wide-brimmed felt hat, ankle boots, and perhaps a delicate scarf that can billow in the wind."
  },
  {
    "instruction": "Suggest a look for a sub-zero winter morning walk",
    "input": "Person: 60 year old non-binary individual, brisk exercise, needs ultimate warmth",
    "output": "Heavy-duty insulated winter coat, thermal base layers, fleece-lined trousers, and extreme-cold winter boots with thick wool socks. A balaclava or neck gaiter, insulated gloves/mittens, and a warm hat."
  },
  {
    "instruction": "Create an outfit for a summer outdoor evening concert",
    "input": "Person: 20 year old male, casual enjoyment, wants relaxed and cool",
    "output": "Chinos or dark denim shorts, a graphic tee or a band t-shirt, and an open lightweight button-up shirt. Comfortable sneakers. A baseball cap and sunglasses for the sunset."
  },
  {
    "instruction": "Recommend attire for a mild autumn afternoon picnic",
    "input": "Person: 28 year old female, casual gathering, likes cozy and charming",
    "output": "A long-sleeved floral midi dress, paired with comfortable ankle boots or stylish flats. A denim jacket or a light cardigan. A wool blanket scarf and a straw or felt hat."
  },
  {
    "instruction": "Suggest a look for a spring office party (cocktail)",
    "input": "Person: 35 year old male, professional networking, wants sharp and festive",
    "output": "A tailored sport coat in a lighter color (e.g., light grey, tan), crisp white dress shirt, and dark dress trousers. A silk pocket square with a subtle pattern. Polished loafers. No tie needed for a 'cocktail' feel."
  },
  {
    "instruction": "Create an outfit for a very hot outdoor wedding",
    "input": "Person: 40 year old female, guest, needs elegant and heat-friendly",
    "output": "A sleeveless, flowy maxi dress in a light, breathable fabric (e.g., cotton voile, linen blend) and bright color or print. Elegant flat sandals or low block heels. Wide-brimmed sun hat or fascinator and minimal jewelry."
  },
  {
    "instruction": "Recommend attire for a cold, windy coastal vacation",
    "input": "Person: 50 year old non-binary individual, exploring, needs warmth and protection",
    "output": "Insulated waterproof jacket, fleece-lined trousers or thick jeans, and waterproof hiking boots. Layer with a warm sweater. A wool beanie, waterproof gloves, and a chunky scarf."
  },
  {
    "instruction": "Suggest a look for a rainy spring workday (hybrid)",
    "input": "Person: 30 year old female, needs to be presentable for video calls and commutes",
    "output": "Dark tailored joggers or comfortable trousers, a neat long-sleeved top or a fine-gauge knit sweater. A stylish waterproof trench coat for commuting. Comfortable, waterproof loafers or chic ankle boots."
  },
  {
    "instruction": "Create an outfit for a humid summer business trip",
    "input": "Person: 45 year old male, client meetings, needs polished and comfortable",
    "output": "Lightweight linen suit in a neutral color (e.g., stone, light grey), a breathable cotton dress shirt (no tie usually required in humid climates). Polished leather loafers. A light briefcase."
  },
  {
    "instruction": "Recommend attire for a dry, dusty desert festival",
    "input": "Person: 22 year old female, active, needs comfort and protection",
    "output": "Loose-fitting linen trousers or cargo shorts, a breathable cotton or linen top. Sturdy closed-toe boots or shoes. A wide-brimmed hat, sunglasses, and a bandana to cover the face from dust. Hydration pack."
  },
  {
    "instruction": "Suggest a look for a cool autumn evening stroll",
    "input": "Person: 65 year old male, leisurely activity, prefers classic and comfortable",
    "output": "Dark wash straight-leg jeans, a long-sleeved polo shirt, and a V-neck merino wool sweater. A quilted jacket or a classic peacoat. Comfortable walking shoes and a warm scarf."
  },
  {
    "instruction": "Create an outfit for a very hot and sunny rooftop party",
    "input": "Person: 28 year old non-binary individual, social event, wants stylish and fun",
    "output": "Colorful tailored shorts or a flowy midi skirt, paired with a stylish tank top or a light camp-collar shirt. Espadrille wedges or fashionable flat sandals. Sunglasses and a statement hat."
  },
  {
    "instruction": "Recommend attire for a freezing winter cabin getaway",
    "input": "Person: 30 year old female, cozy and rustic, needs warmth and comfort",
    "output": "Fleece-lined leggings, an oversized chunky knit sweater, and wool socks. Warm, insulated winter boots. A warm beanie, thick gloves, and a faux fur-lined parka for venturing outdoors."
  },
  {
    "instruction": "Suggest a look for a mild spring outdoor market",
    "input": "Person: 25 year old male, browsing, likes casual and relaxed",
    "output": "Dark wash jeans, a graphic t-shirt, and an open denim jacket or a light bomber. Comfortable sneakers. A baseball cap and a reusable tote bag for purchases."
  },
  {
    "instruction": "Create an outfit for a rainy day at an indoor museum",
    "input": "Person: 50 year old female, cultural outing, wants sophisticated and comfortable",
    "output": "Tailored dark trousers, a silk blouse, and a lightweight cashmere cardigan. Comfortable elegant loafers or low block heels. A chic raincoat or trench coat for arrival/departure, and a stylish umbrella."
  },
  {
    "instruction": "Recommend attire for a sweltering summer outdoor wedding",
    "input": "Person: 35 year old male, guest, needs breathable and formal",
    "output": "A lightweight linen suit in a light color (e.g., light blue, tan), paired with a crisp white or pastel dress shirt (no tie). Polished leather loafers. Sunglasses for sun protection."
  },
  {
    "instruction": "Suggest a look for a chilly spring outdoor cafe",
    "input": "Person: 28 year old non-binary individual, casual meet-up, prefers trendy and warm",
    "output": "Wide-leg jeans, a striped long-sleeved t-shirt, and an oversized knit cardigan. Layer with a stylish trench coat. Comfortable platform sneakers or ankle boots. A scarf for added warmth."
  },
  {
    "instruction": "Create an outfit for a snowy winter commute",
    "input": "Person: 40 year old female, urban professional, needs warmth and durability",
    "output": "Insulated waterproof winter coat, thermal leggings under tailored wool trousers, and warm, waterproof winter boots. A cashmere beanie, insulated gloves, and a thick scarf. A durable work bag."
  },
  {
    "instruction": "Recommend activewear for a mild autumn jog",
    "input": "Person: 30 year old male, light exercise, needs comfortable layers",
    "output": "Running tights or athletic joggers, a moisture-wicking long-sleeved top, and a lightweight running jacket. Running shoes. A light beanie or headband if it's breezy."
  },
  {
    "instruction": "Suggest a look for a hot summer outdoor concert",
    "input": "Person: 25 year old female, enjoying music, needs cool and trendy",
    "output": "Denim shorts, a loose-fitting graphic t-shirt (perhaps tied at the waist), and comfortable sneakers. A baseball cap or bucket hat, sunglasses, and a small crossbody bag for essentials."
  },
  {
    "instruction": "Create an outfit for a freezing winter evening party (indoors)",
    "input": "Person: 50 year old male, semi-formal, wants elegant and warm",
    "output": "Wool blend suit in a dark color (e.g., charcoal, navy), a fine-gauge merino wool turtleneck sweater instead of a shirt and tie. Polished leather dress boots. A classic wool overcoat for arrival."
  },
  {
    "instruction": "Recommend attire for a blustery spring city walk",
    "input": "Person: 65 year old female, leisurely sightseeing, needs warmth and protection",
    "output": "Comfortable dark trousers, a long-sleeved knit top, and a mid-length waterproof trench coat. Comfortable walking shoes. A stylish scarf that can be wrapped around the head or neck for wind protection."
  },
  {
    "instruction": "Suggest a look for a humid summer office event",
    "input": "Person: 30 year old non-binary individual, networking, needs professional and light",
    "output": "Lightweight tailored trousers in a breathable fabric (e.g., linen blend, seersucker), a short-sleeved collared shirt or a smart shell top. Polished loafers or elegant flat sandals. Minimal jewelry."
  },
  {
    "instruction": "Create an outfit for a cold autumn outdoor festival",
    "input": "Person: 22 year old male, enjoying music, wants rugged and warm",
    "output": "Dark jeans, a thermal long-sleeved shirt, and an oversized flannel. Layer with a thick denim jacket or a workwear-style jacket. Sturdy boots. A knit beanie and warm gloves."
  },
  {
    "instruction": "Recommend attire for a sunny spring picnic",
    "input": "Person: 28 year old female, casual outdoor meal, likes cheerful and comfortable",
    "output": "A light floral print midi dress or a skirt and blouse combo, with comfortable flat sandals or white sneakers. A denim jacket for potential breeze. A straw hat and a picnic basket-style bag."
  },
  {
    "instruction": "Suggest a look for a rainy winter indoor activity (bowling)",
    "input": "Person: 20 year old male, fun with friends, needs casual and easy to move in",
    "output": "Comfortable jeans, a graphic t-shirt, and a hoodie. Sneakers (you'll likely change into bowling shoes). A waterproof jacket for arrival/departure. Keep it relaxed and fun."
  },
  {
    "instruction": "Create an outfit for a very hot summer gardening session",
    "input": "Person: 50 year old female, active hobby, needs sun protection and breathability",
    "output": "Loose-fitting cotton gardening trousers or capris, a long-sleeved lightweight cotton shirt (for sun protection), and comfortable gardening clogs or sturdy sandals. A wide-brimmed sun hat, sunglasses, and gardening gloves."
  },
  {
    "instruction": "Recommend attire for a mild autumn business lunch",
    "input": "Person: 35 year old male, client meeting, wants polished and seasonal",
    "output": "Tailored chinos in a fall color (e.g., olive, burgundy), a long-sleeved button-down shirt, and a sport coat in tweed or wool. Polished loafers. A subtle patterned tie can be added for extra formality."
  },
  {
    "instruction": "Suggest a look for a freezing winter outdoor festival (ice sculptures)",
    "input": "Person: 25 year old female, tourist, needs extreme warmth and style",
    "output": "Thermal base layers under insulated snow pants, a chunky knit sweater, and a high-performance winter parka. Warm, waterproof winter boots. A faux fur-lined hat, thick insulated mittens, and a very warm scarf."
  },
  {
    "instruction": "Create an outfit for a humid summer evening walk",
    "input": "Person: 60 year old non-binary individual, leisurely activity, wants light and comfortable",
    "output": "Lightweight linen trousers or capris, a loose-fitting cotton tunic or a short-sleeved camp shirt. Comfortable flat sandals. A light shawl or cardigan if there's a slight breeze. Mosquito repellent is a good idea!"
  },
  {
    "instruction": "Recommend attire for a cold spring outdoor sporting event (soccer game)",
    "input": "Person: 30 year old male, spectator, needs warmth and comfort",
    "output": "Dark jeans or insulated trousers, a long-sleeved t-shirt, and a warm team hoodie. Layer with a windproof and waterproof jacket. A knit beanie, gloves, and sturdy sneakers or boots."
  },
  {
    "instruction": "Suggest a look for a sunny autumn photography walk",
    "input": "Person: 28 year old female, hobbyist, needs comfortable and practical",
    "output": "Comfortable dark jeans or corduroys, a long-sleeved t-shirt, and a practical utility jacket with pockets. Comfortable walking boots or sneakers. A scarf, a beanie (if chilly), and a sturdy camera bag."
  },
  {
    "instruction": "Create an outfit for a very hot summer formal event (garden party)",
    "input": "Person: 45 year old male, guest, needs elegant and heat-friendly",
    "output": "A light-colored linen suit (e.g., cream, sky blue), a breathable cotton dress shirt (no tie), and polished loafers. A pocket square for a touch of elegance. Sunglasses."
  },
  {
    "instruction": "Recommend attire for a misty spring morning commute",
    "input": "Person: 35 year old non-binary individual, professional, needs smart and water-resistant",
    "output": "Tailored dark trousers, a crisp button-down shirt or smart knit top, and a water-resistant trench coat. Comfortable waterproof loafers or sleek ankle boots. A sturdy umbrella and a work bag."
  },
  {
    "instruction": "Suggest a look for a freezing winter business travel",
    "input": "Person: 50 year old female, executive, needs warm, professional, and packable",
    "output": "Tailored wool blend trousers, a cashmere sweater or merino wool turtleneck, and a warm, stylish wool coat. Comfortable yet elegant waterproof winter boots. A warm scarf, gloves, and a structured carry-on bag."
  },
  {
    "instruction": "Create an outfit for a humid summer casual dinner",
    "input": "Person: 25 year old male, social gathering, wants relaxed and smart",
    "output": "Linen blend shorts or light chinos, a short-sleeved polo shirt or a camp collar shirt. Leather sandals or clean sneakers. Minimal accessories."
  },
  {
    "instruction": "Recommend attire for a cold autumn football game (spectator)",
    "input": "Person: 20 year old female, cheering, needs warm and spirited",
    "output": "Dark wash jeans or insulated leggings, a thermal long-sleeved top, and a thick team hoodie. Layer with a warm puffer jacket. A knit beanie, thick gloves, and comfortable warm boots."
  },
  {
    "instruction": "Suggest a look for a sunny spring outdoor market",
    "input": "Person: 30 year old non-binary individual, browsing, likes bohemian and comfortable",
    "output": "A flowy maxi skirt in a natural fabric (e.g., cotton, linen), a simple tank top, and a lightweight cardigan or a tie-dye t-shirt. Comfortable flat sandals or espadrilles. A wide-brimmed straw hat and a large canvas tote bag."
  },
  {
    "instruction": "Create an outfit for a rainy day at a home office",
    "input": "Person: 40 year old female, working from home, wants cozy and presentable",
    "output": "Soft knit joggers, an oversized but stylish knit sweater, and comfortable slippers. A warm blanket draped nearby. Keep hair casually styled for video calls."
  },
  {
    "instruction": "Recommend attire for a very hot outdoor casual gathering",
    "input": "Person: 35 year old male, backyard BBQ, needs light and relaxed",
    "output": "Linen shorts, a breathable cotton t-shirt or a Hawaiian shirt, and comfortable sandals or canvas sneakers. Sunglasses and a baseball cap."
  },
  {
    "instruction": "Suggest a look for a snowy winter evening stroll",
    "input": "Person: 60 year old female, leisurely activity, needs maximum warmth",
    "output": "Fleece-lined trousers or insulated leggings under a midi skirt, a cashmere turtleneck, and a knee-length insulated winter coat. Warm, waterproof winter boots. A faux fur-lined hat, warm mittens, and a thick scarf."
  },
  {
    "instruction": "Create an outfit for a humid summer art gallery opening",
    "input": "Person: 28 year old female, evening event, wants sophisticated and cool",
    "output": "A sleek sleeveless jumpsuit in a breathable fabric (e.g., Tencel, silk blend), or a flowy midi dress. Elegant flat sandals or low block heels. Minimalist jewelry and a small clutch."
  },
  {
    "instruction": "Recommend attire for a cold autumn evening date",
    "input": "Person: 25 year old male, dinner and drinks, wants stylish and warm",
    "output": "Dark wash slim-fit jeans, a fine-gauge knit sweater in a rich color, and a stylish bomber jacket or a wool blazer. Chukka boots or polished leather sneakers. A subtle scarf for extra warmth."
  },
  {
    "instruction": "Suggest a look for a sunny spring outdoor wedding",
    "input": "Person: 32 year old non-binary individual, guest, prefers cheerful and smart",
    "output": "A pastel-colored suit (e.g., light blue, mint green) with a white button-down shirt or a floral blouse. Polished loafers or elegant flat sandals. A colorful pocket square or a delicate brooch."
  },
  {
    "instruction": "Create an outfit for a very hot summer theme park visit",
    "input": "Person: 20 year old female, active day, needs ultimate comfort and sun protection",
    "output": "Athletic shorts, a breathable tank top or t-shirt, and supportive sneakers. A wide-brimmed sun hat, sunglasses, and a small backpack with water. Sunscreen is crucial!"
  },
  {
    "instruction": "Recommend attire for a rainy autumn evening at home",
    "input": "Person: 30 year old male, relaxing, wants cozy and comfortable",
    "output": "Soft sweatpants or flannel pajama bottoms, an oversized hoodie or a cozy long-sleeved t-shirt. Warm wool socks. A comfortable blanket and a hot beverage."
  },
  {
    "instruction": "Suggest a look for a freezing winter casual gathering",
    "input": "Person: 40 year old female, friends' house, needs warmth and comfort",
    "output": "Fleece-lined leggings or insulated dark jeans, a thick, oversized knit sweater, and wool socks paired with warm indoor slippers. Layer with a puffer vest or a cozy cardigan. A warm beanie for travel."
  },
  {
    "instruction": "Create an outfit for a humid summer outdoor dining",
    "input": "Person: 50 year old male, casual upscale, needs breathable and smart",
    "output": "Lightweight linen trousers in a neutral color, a short-sleeved button-up shirt in a light print or solid. Leather loafers or elegant boat shoes. A stylish watch."
  },
  {
    "instruction": "Recommend attire for a cold spring outdoor market",
    "input": "Person: 25 year old non-binary individual, browsing, wants practical and stylish",
    "output": "Dark wash jeans, a long-sleeved thermal top, and a chunky knit sweater. Layer with a stylish utility jacket or a warm denim jacket. Comfortable waterproof ankle boots. A knit beanie and a large canvas tote."
  },
  {
    "instruction": "Suggest a look for a sunny autumn outdoor yoga class",
    "input": "Person: 35 year old female, active, needs flexible and comfortable",
    "output": "High-waisted yoga leggings, a breathable long-sleeved top (for sun and slight chill), and a light fleece or hoodie for warm-up/cool-down. Barefoot or yoga socks. Sunglasses."
  },
  {
    "instruction": "Create an outfit for a very hot summer evening party (casual)",
    "input": "Person: 28 year old male, social event, wants relaxed and cool",
    "output": "Tailored linen shorts, a stylish short-sleeved button-up shirt (e.g., resort shirt), and comfortable espadrilles or leather sandals. A light watch."
  },
  {
    "instruction": "Recommend attire for a rainy winter indoor concert",
    "input": "Person: 22 year old female, music enthusiast, needs comfort and waterproof outer layer",
    "output": "Dark wash jeans or black trousers, a band t-shirt, and a stylish bomber jacket or a denim jacket. Comfortable sneakers. A waterproof trench coat for entry/exit, and a sturdy umbrella."
  },
  {
    "instruction": "Suggest a look for a freezing winter outdoor ice skating",
    "input": "Person: 20 year old non-binary individual, recreational, needs extreme warmth and freedom of movement",
    "output": "Thermal base layers, fleece-lined leggings or snow pants, a warm sweater, and a waterproof/insulated ski jacket. Warm, waterproof gloves or mittens, a thick knit beanie, and warm wool socks. Skates are typically rented!"
  },
  {
    "instruction": "Create an outfit for a humid summer professional conference",
    "input": "Person: 40 year old male, presenting, needs polished and comfortable",
    "output": "Lightweight wool or linen blend suit in a light color (e.g., light grey, blue), a breathable cotton dress shirt, and a light silk tie (optional). Polished leather loafers. A light briefcase."
  },
  {
    "instruction": "Recommend attire for a cold autumn hiking trip",
    "input": "Person: 30 year old female, active, needs layered and durable",
    "output": "Moisture-wicking base layers, fleece-lined hiking pants, a warm fleece jacket, and a waterproof/windproof outer shell jacket. Hiking boots, wool socks, a beanie, and gloves."
  },
  {
    "instruction": "Suggest a look for a sunny spring outdoor cafe lunch",
    "input": "Person: 55 year old female, leisurely meal, prefers elegant and comfortable",
    "output": "Tailored capris or a midi skirt in a light color, a patterned silk blouse, and a lightweight cardigan. Comfortable elegant flats or low wedges. Sunglasses and a stylish handbag."
  },
  {
    "instruction": "Create an outfit for a very hot summer casual event (rooftop drinks)",
    "input": "Person: 28 year old non-binary individual, social, wants trendy and breathable",
    "output": "Wide-leg linen trousers or a flowy midi skirt, a stylish cropped top or a breathable camisole. Flat sandals or espadrilles. Sunglasses and a statement hat."
  },
      {
        "instruction": "Recommend an outfit for a casual summer day",
        "input": "Person: 25 year old female, prefers comfortable clothing, loves pastel colors",
        "output": "I recommend a light pink linen shirt paired with white denim shorts and beige sandals. Add a straw hat and a small crossbody bag in cream color to complete the look."
    },
    {
        "instruction": "Suggest formal office wear",
        "input": "Person: 30 year old male, professional setting, prefers classic styles",
        "output": "A navy blue tailored suit with a crisp white dress shirt would be perfect. Pair it with a burgundy tie, black oxford shoes, and a matching leather belt."
    },
    {
        "instruction": "Create a date night outfit",
        "input": "Person: 28 year old female, romantic dinner, likes elegant styles",
        "output": "I suggest a black midi dress with a subtle floral pattern, paired with nude heels and minimal gold jewelry. Add a small clutch and a light cardigan for the evening."
    },
    {
        "instruction": "Recommend gym workout clothes",
        "input": "Person: 22 year old female, high-intensity training, prefers bold colors",
        "output": "Go for bright red high-waisted leggings with a black sports bra and a matching red cropped hoodie. Pair with white training sneakers and add a black gym bag."
    },
    {
        "instruction": "Suggest winter street style",
        "input": "Person: 26 year old male, urban lifestyle, loves streetwear",
        "output": "Layer a black hoodie under a gray puffer jacket, pair with dark blue jeans and white sneakers. Add a black beanie and a crossbody bag for the complete street look."
    },
    {
        "instruction": "Create a beach vacation outfit",
        "input": "Person: 24 year old female, tropical vacation, loves vibrant patterns",
        "output": "A colorful floral maxi dress in turquoise and coral tones, paired with tan leather sandals. Add oversized sunglasses, a woven beach bag, and a wide-brim straw hat."
    },
    {
        "instruction": "Recommend business casual attire",
        "input": "Person: 35 year old female, creative industry, prefers modern styles",
        "output": "Wear a camel-colored blazer over a white silk blouse with black tailored trousers. Pair with pointed-toe ankle boots and add a structured leather tote bag."
    },
    {
        "instruction": "Suggest weekend brunch outfit",
        "input": "Person: 27 year old male, relaxed setting, casual style",
        "output": "A light blue button-down shirt (sleeves rolled up) with khaki chinos and brown loafers. Add a brown leather belt and aviator sunglasses for a polished casual look."
    },
    {
        "instruction": "Create a concert outfit",
        "input": "Person: 23 year old female, rock concert, edgy style preferred",
        "output": "Black leather pants with a band graphic tee and a denim jacket. Pair with black combat boots, silver chain accessories, and a small black backpack."
    },
    {
        "instruction": "Recommend fall layering outfit",
        "input": "Person: 29 year old male, transitional weather, minimalist style",
        "output": "A charcoal gray crewneck sweater over a white t-shirt with dark wash jeans. Layer with a tan trench coat and finish with brown Chelsea boots."
    }
]


🎨 Creating fashion recommendation dataset...


In [ ]:
# Format data for instruction fine-tuning
def format_instruction(sample):
    return f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""

formatted_data = [{"text": format_instruction(item)} for item in fashion_data]

# Create dataset
dataset = Dataset.from_list(formatted_data)
print(f"✅ Created dataset with {len(dataset)} examples")


✅ Created dataset with 275 examples


In [ ]:
# ============================================================================
# STEP 4: Load Base Model and Tokenizer
# ============================================================================
print("\n🤖 Loading base model...")

# BEST OPEN-SOURCE OPTIONS (No authentication required):
model_name = "Qwen/Qwen2-1.5B-Instruct"  # Fast, efficient, great for fine-tuning

# Other excellent alternatives (uncomment to use):
# model_name = "microsoft/phi-2"  # 2.7B - Very smart, good for reasoning
# model_name = "Qwen/Qwen2-1.5B-Instruct"  # 1.5B - Excellent instruction following
# model_name = "stabilityai/stablelm-2-1_6b"  # 1.6B - Stable AI's model
# model_name = "HuggingFaceH4/zephyr-7b-beta"  # 7B - More powerful but slower

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("✅ Model and tokenizer loaded successfully!")

# ============================================================================
# STEP 5: Configure LoRA
# ============================================================================
print("\n⚙️ Configuring LoRA...")

lora_config = LoraConfig(
    r=16,                      # LoRA rank
    lora_alpha=32,             # LoRA alpha scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Target attention layers
    lora_dropout=0.05,         # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA configuration applied!")

# ============================================================================
# STEP 6: Setup Training Arguments
# ============================================================================
print("\n🏋️ Setting up training configuration...")

training_args = TrainingArguments(
    output_dir="./fashion-recommender-model-test",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    save_steps=50,
    logging_steps=10,
    save_total_limit=2,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    report_to="none",
)



🤖 Loading base model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model and tokenizer loaded successfully!

⚙️ Configuring LoRA...
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
✅ LoRA configuration applied!

🏋️ Setting up training configuration...


In [ ]:
# ============================================================================
# STEP 7: Tokenize Dataset
# ============================================================================
print("\n📝 Tokenizing dataset...")

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

print("✅ Dataset tokenized!")
# ============================================================================
# STEP 8: Initialize Trainer
# ============================================================================
print("\n🚀 Initializing trainer...")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)
# ============================================================================
# STEP 9: Start Training
# ============================================================================
print("\n🎯 Starting training...\n")
print("=" * 60)

trainer.train()

print("\n✅ Training completed!")





📝 Tokenizing dataset...


Map:   0%|          | 0/275 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


✅ Dataset tokenized!

🚀 Initializing trainer...

🎯 Starting training...



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,2.787500
20,2.050000
30,1.650000
40,1.487500
50,1.375000
60,1.356200
70,1.262500
80,1.168700
90,1.162500
100,1.112500


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



✅ Training completed!


In [ ]:
# ============================================================================
# STEP 10: Save the Fine-tuned Model
# ============================================================================
print("\n💾 Saving model...")

model.save_pretrained("./fashion-recommender-lora-test", safe_serialization=False)
tokenizer.save_pretrained("./fashion-recommender-lora-test")

print("✅ Model saved to './fashion-recommender-lora'")



💾 Saving model...
✅ Model saved to './fashion-recommender-lora'


In [ ]:
# # ============================================================================
# # STEP 11: Test the Model
# # ============================================================================
# print("\n🧪 Testing the fine-tuned model...\n")

# def generate_recommendation(instruction, user_input):
#     prompt = f"""### Instruction:
# {instruction}

# ### Input:
# {user_input}

# ### Response:
# """

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=150,
#             temperature=0.7,
#             top_p=0.9,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id
#         )

#     response = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     # Extract only the response part
#     response = response.split("### Response:")[-1].strip()

#     return response

# # Test examples
# test_cases = [
#     {
#         "instruction": "Recommend an outfit for a wedding guest",
#         "input": "Person: 32 year old female, outdoor garden wedding, elegant style"
#     },
#     {
#         "instruction": "Suggest everyday casual wear",
#         "input": "Person: 26 year old male, work from home, comfortable style"
#     }
# ]

# print("📋 Test Results:")
# print("=" * 60)

# for i, test in enumerate(test_cases, 1):
#     print(f"\n🔹 Test {i}:")
#     print(f"Instruction: {test['instruction']}")
#     print(f"Input: {test['input']}")
#     recommendation = generate_recommendation(test['instruction'], test['input'])
#     print(f"Recommendation: {recommendation}")
#     print("-" * 60)
# # Step 9: Inference example (test the model)
from transformers import pipeline
from transformers import AutoTokenizer
model_name = 'Qwen/Qwen2-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.save_pretrained('./fashion-recommender-lora-test')
generator = pipeline('text-generation', model='./fashion-recommender-lora-test', tokenizer=tokenizer)

prompt = f"""### Instruction:
Suggest gym  outfit
### Input:
Person: young boy
### Response:
"""
result = generator(prompt, max_new_tokens=200, num_return_sequences=1)
print(result[0]['generated_text'])



Device set to use cpu


### Instruction:
Suggest gym  outfit
### Input:
Person: young boy
### Response:
Sports jersey or a graphic t-shirt, athletic shorts, and sneakers. A baseball cap can be added for a casual look.


In [ ]:
# ============================================================================
# STEP 12: Interactive Chat Interface
# ============================================================================
print("\n💬 Starting interactive chat mode...")
print("Type 'quit' to exit\n")

def chat():
    print("Fashion Stylist AI - Ready to help! 🎨")
    print("-" * 60)

    while True:
        user_desc = input("\n👤 Describe the person and occasion (or 'quit'): ")

        if user_desc.lower() == 'quit':
            print("Thanks for using Fashion Stylist AI! 👋")
            break

        instruction = "Recommend a complete outfit"
        recommendation = generate_recommendation(instruction, f"Person: {user_desc}")

        print(f"\n✨ Recommendation:\n{recommendation}")
        print("-" * 60)

# Uncomment to start interactive chat
chat()
